Esse Notebook tem como objetivo:\
•Criar o banco de dados (database) da camada Bronze.\
• Criar as tabelas Bronze a partir de cada CSV, lendo os dados (sem qualquer alteração estrutural ou de
conteúdo).\
• Adicionar em cada tabela a coluna ingestion_datetime, contendo o timestamp exato do momento da
inserção do dado na camada Bronze.\
• Gravar todas as tabelas em formato Delta, utilizando o modo Append.\
• Ingestão de API, com os devidos requisitos da atividade.

In [0]:
dbutils.widgets.text("data_inicio", "")  
dbutils.widgets.text("data_fim", "")

data_inicio_widget = dbutils.widgets.get("data_inicio")  
data_fim_widget = dbutils.widgets.get("data_fim")


Os widgets, nos permitem parametrização manual ou via Job.


Quando o Job roda o notebook automaticamente, ele pode pegar essas datas como parâmetro, sem precisar digitar nada manualmente.

In [0]:
from datetime import datetime, timedelta

# Se o widget estiver vazio, vai pegar os últimos 7 dias automaticamente
if data_inicio_widget == "":
    hoje = datetime.now()
    sete_dias_atras = hoje - timedelta(days=7)
    data_fim_formatada = hoje.strftime("%m-%d-%Y")
    data_inicio_formatada = sete_dias_atras.strftime("%m-%d-%Y")
else:
    data_inicio_formatada = data_inicio_widget
    data_fim_formatada = data_fim_widget



In [0]:
import requests

url = f"https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/CotacaoDolarPeriodo(dataInicial=@dataInicial,dataFinalCotacao=@dataFinalCotacao)?@dataInicial='{data_inicio_formatada}'&@dataFinalCotacao='{data_fim_formatada}'&$select=dataHoraCotacao,cotacaoCompra&$format=json"

#vamos chamar a API
resposta = requests.get(url)
dados_json = resposta.json()
lista_cotacoes = dados_json["value"]

In [0]:
from pyspark.sql.functions import current_timestamp
df_cotacao = spark.createDataFrame(lista_cotacoes)
df_cotacoes_bronze = df_cotacao.withColumn("ingestion_datetime",current_timestamp())

In [0]:
#display(df_cotacoes_bronze)

In [0]:
(df_cotacoes_bronze.write
    .mode("append")
    .format("delta")
    .option("mergeSchema", "true")
    .saveAsTable("cinedata.bronze.tb_cotacao_dolar")
)

A ideia agora é fazer a carga já com os nomes mapeados e de umA só vez utilizando uma função.

A gente pode usar o "dbutils.fs.ls" para poder ver as infomações do folder e com isso pegar o nome de caminho de cada arquivo.

In [0]:
from pyspark.sql.functions import current_timestamp

caminho_origem = "/Volumes/cinedata/landing/inputs/"

mapeamento = {
    "movies_info_TMDB_IMDB.csv": "tb_movies_info",   # <- corrigido: TMDB antes de IMDB
    "movies_financials_IMDB_TMDB.csv": "tb_movies_financials",
    "movies_metrics_IMDB_TMDB.csv": "tb_movies_metrics",
    "credits_and_tags_IMDB_TMDB.csv": "tb_credits_and_tags",
    "movies_reviews.csv": "tb_movies_reviews",
}

def carregar_bronze(caminho_arquivo, nome_tabela, catalog="cinedata", schema="bronze"):
    df = spark.read.csv(caminho_arquivo, header=True, inferSchema=True)
    df_bronze = df.withColumn("ingestion_datetime", current_timestamp())
    (df_bronze.write
        .mode("append")
        .format("delta")
        .option("mergeSchema", "true")
        .saveAsTable(f"{catalog}.{schema}.{nome_tabela}")
    )
    print(f"{nome_tabela} carregada")

arquivos_no_volume = dbutils.fs.ls(caminho_origem)

#Aqui pegamos as informações de todos os arquivos no volume, para fazermos o mapeamento dos nomes e a carga com a função
for arquivo in arquivos_no_volume:
    nome_arquivo = arquivo.name
    if nome_arquivo in mapeamento:
        nome_tabela = mapeamento[nome_arquivo]
        try:
            carregar_bronze(arquivo.path, nome_tabela)
        except Exception as e:
            print(f"Erro ao carregar {nome_tabela}: {e}")
    else:
        print(f"Ignorado (fora do mapeamento): {nome_arquivo}")

tb_credits_and_tags carregada
tb_movies_financials carregada
tb_movies_info carregada
tb_movies_metrics carregada
tb_movies_reviews carregada


# Guia das Células do Notebook

| Célula | Tipo | O que faz |
| --- | --- | --- |
| 1 | Markdown | Introdução do notebook — descreve os objetivos: criar database bronze, carregar CSVs sem alteração, adicionar `ingestion_datetime`, gravar em Delta (append) e ingerir dados de API. |
| 2 | Python | Cria dois widgets de texto (`data_inicio` e `data_fim`) para parametrizar o período da consulta à API de cotação do dólar. |
| 3 | Markdown | Explica que os widgets permitem parametrização manual ou via Job — o Job pode passar as datas automaticamente sem digitação. |
| 4 | Python | Se os widgets estiverem vazios, calcula automaticamente os últimos 7 dias (`hoje` e `hoje - 7`) e formata as datas no padrão `MM-DD-YYYY`. Caso contrário, usa os valores informados. |
| 5 | Python | Monta a URL da API do Banco Central (PTAX) com as datas e faz a requisição HTTP `GET`; extrai a lista de cotações do JSON retornado. |
| 6 | Python | Converte a lista de cotações em um DataFrame Spark e adiciona a coluna `ingestion_datetime` com `current_timestamp()`. |
| 7 | Python | `display(df_cotacoes_bronze)` comentado — serve para inspecionar visualmente o DataFrame durante o desenvolvimento. |
| 8 | Python | Salva o DataFrame da cotação do dólar como tabela Delta `cinedata.bronze.tb_cotacao_dolar` em modo append, permitindo evolução de schema. |
| 9 | Markdown | Explica a estratégia de carga dos CSVs: usar um dicionário de mapeamento (nome do arquivo → nome da tabela) e `dbutils.fs.ls` para listar os arquivos do volume. |
| 10 | Python | Define a função `carregar_bronze` (lê CSV, adiciona `ingestion_datetime`, salva em Delta append), lista os arquivos do volume com `dbutils.fs.ls` e itera sobre eles carregando cada CSV mapeado para sua tabela bronze correspondente. |